# Лабораторная работа №5

## Генерация тестовых данных

In [121]:
import numpy as np
import random

In [122]:
VEC_LEN = 4
NUM_CALLS = 100000

In [123]:
def generate_test_data(seed=740):
    random.seed(seed)
    np.random.seed(seed)
    
    vectors = []
    for i in range(NUM_CALLS + 1):
        vec = tuple(random.uniform(-1000, 1000) for _ in range(VEC_LEN))
        vectors.append(vec)
    
    pairs = [(vectors[i], vectors[i+1]) for i in range(NUM_CALLS)]
    return pairs

In [124]:
pairs = generate_test_data()

## Ctypes

In [125]:
import ctypes
import os

In [126]:
class DotResult(ctypes.Structure):
    _fields_ = [
        ("result", ctypes.c_double),
        ("error", ctypes.c_int)
    ]

lib_path = os.path.abspath("libvector_dot.dylib")
lib = ctypes.CDLL(lib_path)

lib.dot_product.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double)]
lib.dot_product.restype = DotResult

def dot_product_ctypes(a, b):
    a_arr = (ctypes.c_double * 4)(*a)
    b_arr = (ctypes.c_double * 4)(*b)
    
    res = lib.dot_product(a_arr, b_arr)
    
    if res.error:
        raise ValueError("NULL pointer in C function")
    return res.result

def run_test_ctypes():
    results = []
    for a, b in pairs:
        results.append(dot_product_ctypes(a, b))
    return results


## CFFI

In [127]:
from cffi import FFI

ffi = FFI()

ffi.cdef("""
    #define VEC_LEN 4
    typedef struct {
        double result;
        int error;
    } DotResult;
    
    DotResult dot_product(const double *a, const double *b);
""")

lib_cffi = ffi.dlopen('./libvector_dot.dylib')

def dot_product_cffi(a, b):
    a_ptr = ffi.new("double[]", a)
    b_ptr = ffi.new("double[]", b)
    result = lib_cffi.dot_product(a_ptr, b_ptr)
    if result.error:
        raise ValueError("NULL pointer in C function")
    return result.result

def run_test_cffi():
    results = []
    for a, b in pairs:
        results.append(dot_product_cffi(a, b))
    return results

## CPython

In [128]:
import vector_lib

def run_test_cpython():
    results = []
    for a, b in pairs:
        results.append(vector_lib.dot_product(list(a), list(b)))
    return results

## Cython

In [129]:
import array
from vector_wrapper import py_dot_product

def run_test_cython():
    results = []
    for a, b in pairs:
        a_arr = array.array('d', a)
        b_arr = array.array('d', b)
        results.append(py_dot_product(a_arr, b_arr))
    return results

## Тестирование

In [130]:
import time
import statistics

def benchmark(impl_name, run_func, num_runs=50):
    print(f"Benchmarking {impl_name}...")
    times = []
    
    for i in range(num_runs):
        start = time.perf_counter()
        run_func()
        end = time.perf_counter()
        elapsed = end - start
        times.append(elapsed)
    
    mean = statistics.mean(times)
    median = statistics.median(times)
    std_dev = statistics.stdev(times)
    min_time = min(times)
    max_time = max(times)
    
    return {
        'min': min_time,
        'max': max_time,
        'mean': mean,
        'median': median,
        'std_dev': std_dev
    }

In [131]:
import pandas as pd

def run_comparison(functions_to_test):
    all_results = []
    
    for name, func in functions_to_test.items():
        stats = benchmark(name, func) 
        all_results.append({
            'Способ': name,
            'Медиана (s)': stats['median'],
            'Среднее (s)': stats['mean'],
            'Min (s)': stats['min'],
            'Std Dev': stats['std_dev']
        })
    
    df = pd.DataFrame(all_results)
    
    display(df)

In [132]:
functions_to_test = {"Ctypes": run_test_ctypes,"CFFI": run_test_cffi, "CPython": run_test_cpython, "Cython": run_test_cython}
run_comparison(functions_to_test)

Benchmarking Ctypes...
Benchmarking CFFI...
Benchmarking CPython...
Benchmarking Cython...


,Способ,Медиана (s),Среднее (s),Min (s),Std Dev
0,Ctypes,0.105254,0.105714,0.104370,0.001346
1,CFFI,0.052614,0.053078,0.052397,0.001016
2,CPython,0.012253,0.012283,0.011965,0.000172
3,Cython,0.039214,0.039423,0.037975,0.000658
